# 06_mock: SQL + Python Data Cleaning (Practical Applied Task)

Timebox: **55 minutes**  
Language: **Python (Colab)**

## Scenario
You are given a demo transactional table with duplicates and dirty fields. Build a clean extraction pipeline using SQL and Python.

## What to implement
1. `parse_amount`
2. `parse_order_date`
3. `extract_clean_rows`
4. `summarize_by_region`
5. `top_day`

## Completion criteria (required)
- Latest-row dedupe per order (`updated_at`)
- Correct amount/date normalization
- Invalid-row filtering
- Correct aggregated metrics

## Time guidance
- 10 min: identify required cleaning rules and failure cases
- 35 min: implement SQL extraction plus Python normalization
- 10 min: run tests and verify metric correctness

## Interviewer follow-up questions (prepare answers)
1. Why did you split logic between SQL and Python the way you did?
2. How do you ensure deterministic dedupe when there are ties in `updated_at`?
3. Which malformed amount/date inputs are still unsupported, and why?
4. What data-quality metrics would you emit to monitor pipeline health over time?
5. If dataset size grows 100x, what parts should be pushed down into SQL for performance?


In [ ]:
import sqlite3
from collections import defaultdict
from datetime import datetime
from typing import Any

RAW_SALES = [
    ("o1", "2025-01-02", "us", "$1,200.00", "2025-01-02T10:00:00"),
    ("o1", "2025-01-02", "US", "$1,250.00", "2025-01-02T12:00:00"),  # latest wins
    ("o2", "01/03/2025", " eu ", "850", "2025-01-03T09:00:00"),
    ("o3", "2025-01-03", "", "N/A", "2025-01-03T09:30:00"),  # invalid amount -> drop
    ("o4", "2025-01-04", "apac", "300.5", "2025-01-04T08:00:00"),
    ("o5", "2025-13-04", "us", "100", "2025-01-04T08:00:00"),  # invalid date -> drop
    ("o6", "2025-01-04", None, " 99.50 ", "2025-01-04T10:00:00"),
]


def make_connection() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    conn.execute(
        """
        CREATE TABLE sales_raw (
            order_id TEXT NOT NULL,
            order_date TEXT NOT NULL,
            region TEXT,
            amount_text TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )
        """
    )
    conn.executemany(
        "INSERT INTO sales_raw (order_id, order_date, region, amount_text, updated_at) VALUES (?, ?, ?, ?, ?)",
        RAW_SALES,
    )
    conn.commit()
    return conn


In [ ]:
def parse_amount(amount_text: str) -> float | None:
    """Parse strings like '$1,200.00' or ' 99.50 ' into float. Return None for invalid values."""
    # TODO
    raise NotImplementedError


def parse_order_date(raw_date: str) -> str | None:
    """Normalize date into YYYY-MM-DD from accepted formats (%Y-%m-%d, %m/%d/%Y)."""
    # TODO
    raise NotImplementedError


def extract_clean_rows(conn: sqlite3.Connection) -> list[dict[str, Any]]:
    """
    Use SQL + Python cleaning rules:
    - Keep latest record per order_id (max updated_at).
    - Parse amount/date.
    - Normalize region to uppercase; blank/null -> UNKNOWN.
    - Drop rows with invalid amount/date or non-positive amount.
    - Return rows sorted by order_id.
    """
    # TODO
    raise NotImplementedError


def summarize_by_region(rows: list[dict[str, Any]]) -> dict[str, float]:
    """Return region -> total amount rounded to 2 decimals."""
    # TODO
    raise NotImplementedError


def top_day(rows: list[dict[str, Any]]) -> tuple[str, float]:
    """Return (date, total_amount) for highest-revenue day; tie -> earliest date."""
    # TODO
    raise NotImplementedError


## Run Tests
Run this final test cell after implementing all TODO sections.


In [ ]:
def run_exam06_tests() -> None:
    conn = make_connection()
    rows = extract_clean_rows(conn)

    assert [row["order_id"] for row in rows] == ["o1", "o2", "o4", "o6"]
    assert rows[0]["amount"] == 1250.0  # latest o1 row wins
    assert rows[1]["order_date"] == "2025-01-03"
    assert rows[3]["region"] == "UNKNOWN"

    summary = summarize_by_region(rows)
    assert summary == {
        "APAC": 300.5,
        "EU": 850.0,
        "UNKNOWN": 99.5,
        "US": 1250.0,
    }

    day, total = top_day(rows)
    assert day == "2025-01-02"
    assert total == 1250.0

    assert parse_amount("N/A") is None
    assert parse_order_date("2025-13-01") is None

    print("06_mock tests passed")


run_exam06_tests()
